In [1]:
import os

In [2]:
%pwd

'c:\\Projects\\End-to-End-ML-Classification-Project-with-MLflow-\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'c:\\Projects\\End-to-End-ML-Classification-Project-with-MLflow-'

In [22]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class ModelTrainerConfig:
    root_dir: Path
    train_data_path: Path
    test_data_path: Path
    train_target_data_path: Path
    test_target_data_path: Path
    model_name: str
    alpha: float
    l1_ratio: float
    target_column: str

In [23]:
from mlProject.constants import *
from mlProject.utils.common import read_yaml, create_directories

In [24]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH,
        schema_filepath = SCHEMA_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])


    def get_model_trainer_config(self) -> ModelTrainerConfig:
        config = self.config.model_trainer
        params = self.params.ElasticNet
        schema =  self.schema.TARGET_COLUMN

        create_directories([config.root_dir])

        model_trainer_config = ModelTrainerConfig(
            root_dir=config.root_dir,
            train_data_path = config.train_data_path,
            test_data_path = config.test_data_path,
            train_target_data_path = config.train_target_data_path,
            test_target_data_path = config.test_target_data_path,
            model_name = config.model_name,
            alpha = params.alpha,
            l1_ratio = params.l1_ratio,
            target_column = schema.name
            
        )

        return model_trainer_config

In [25]:
import pandas as pd
import os
from mlProject import logger
from sklearn.linear_model import ElasticNet
from sklearn.ensemble import RandomForestClassifier
import joblib

In [ ]:
class ModelTrainer:
    def __init__(self, config: ModelTrainerConfig):
        self.config = config

    
    def train(self):
        
        train_x = pd.read_csv(self.config.train_data_path)
        test_x = pd.read_csv(self.config.test_data_path)
        # The target column is read as a dataframe on converted back to a series using the iloc[:,0]
        train_y = pd.read_csv(self.config.train_target_data_path).iloc[:,0]
        test_y = pd.read_csv(self.config.test_target_data_path).iloc[:,0]

        
        rf = RandomForestClassifier()
        rf.fit(train_x, train_y)

        joblib.dump(rf, os.path.join(self.config.root_dir, self.config.model_name))



In [27]:
try:
    config = ConfigurationManager()
    model_trainer_config = config.get_model_trainer_config()
    model_trainer_config = ModelTrainer(config=model_trainer_config)
    model_trainer_config.train()
except Exception as e:
    raise e

[2026-02-25 04:16:17,310: INFO: common: created directory at: artifacts]
[2026-02-25 04:16:17,347: INFO: common: created directory at: artifacts/model_trainer]


c:\Projects\End-to-End-ML-Classification-Project-with-MLflow-\venv\Lib\site-packages\sklearn\base.py:1336: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
